In [2]:
import torch.nn as nn
import torch
import torch.nn.functional as F

torch.cuda.is_available()

True

In [14]:
VOCAB = 20
D_MODEL = 20
batch, seq_len = 2, 4

In [15]:
emb = nn.Embedding(VOCAB, D_MODEL)
wQ = nn.Linear(D_MODEL, D_MODEL)
wK = nn.Linear(D_MODEL, D_MODEL)
wV = nn.Linear(D_MODEL, D_MODEL)

In [17]:
tokens = torch.randint(0, VOCAB, (batch, seq_len))

In [18]:
x = emb(tokens)
tokens.shape, x.shape

(torch.Size([2, 4]), torch.Size([2, 4, 20]))

In [19]:
q = wQ(x)
k = wK(x)
v = wV(x)

In [20]:
q.shape, k.shape, v.shape

(torch.Size([2, 4, 20]), torch.Size([2, 4, 20]), torch.Size([2, 4, 20]))

In [22]:
k.shape, k.transpose(-2, -1).shape

(torch.Size([2, 4, 20]), torch.Size([2, 20, 4]))

In [24]:
(q @ k.transpose(-2, -1)).shape

torch.Size([2, 4, 4])

In [25]:
scores = torch.matmul(q, k.transpose(-2, -1))
print("raw scores:", scores.shape)

scaled_scores = scores / (D_MODEL ** 0.5)
attn_weights = scaled_scores.softmax(dim=-1)
print("attn_weights:", attn_weights.shape)
print(attn_weights[0])
print("rows sum to 1?", attn_weights[0].sum(dim=-1))


raw scores: torch.Size([2, 4, 4])
attn_weights: torch.Size([2, 4, 4])
tensor([[0.2336, 0.1907, 0.1911, 0.3846],
        [0.2935, 0.2121, 0.1609, 0.3335],
        [0.3139, 0.2296, 0.2194, 0.2370],
        [0.1672, 0.2372, 0.2828, 0.3129]], grad_fn=<SelectBackward0>)
rows sum to 1? tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [26]:
output = torch.matmul(attn_weights, v)
print("output:", output.shape)

output: torch.Size([2, 4, 20])


In [28]:
manual_output_00 = (attn_weights[0, 0].unsqueeze(-1) * v[0]).sum(dim=0)
print(torch.allclose(manual_output_00, output[0, 0], atol=1e-6))

True


In [29]:
n_heads = 4
head_dim = D_MODEL // n_heads
print("head_dim:", head_dim)

q_heads = q.view(batch, seq_len, n_heads, head_dim)
k_heads = k.view(batch, seq_len, n_heads, head_dim)
v_heads = v.view(batch, seq_len, n_heads, head_dim)
print("q_heads after view:", q_heads.shape)

q_heads = q_heads.transpose(1, 2)
k_heads = k_heads.transpose(1, 2)
v_heads = v_heads.transpose(1, 2)
print("q_heads after transpose:", q_heads.shape)


head_dim: 5
q_heads after view: torch.Size([2, 4, 4, 5])
q_heads after transpose: torch.Size([2, 4, 4, 5])


In [30]:
scores = torch.matmul(q_heads, k_heads.transpose(-2, -1))
print("scores:", scores.shape)

scaled_scores = scores / (head_dim ** 0.5)
attn_weights = scaled_scores.softmax(dim=-1)
print("attn_weights:", attn_weights.shape)

output_heads = torch.matmul(attn_weights, v_heads)
print("output_heads:", output_heads.shape)


scores: torch.Size([2, 4, 4, 4])
attn_weights: torch.Size([2, 4, 4, 4])
output_heads: torch.Size([2, 4, 4, 5])


In [31]:
output_heads = output_heads.transpose(1, 2)
print("output_heads after transpose:", output_heads.shape)

output_heads = output_heads.contiguous()
output = output_heads.view(batch, seq_len, D_MODEL)
print("output merged:", output.shape)


output_heads after transpose: torch.Size([2, 4, 4, 5])
output merged: torch.Size([2, 4, 20])


In [32]:
wO = nn.Linear(D_MODEL, D_MODEL)
final_output = wO(output)
print("final_output:", final_output.shape)


final_output: torch.Size([2, 4, 20])


In [33]:
x.shape, final_output.shape

(torch.Size([2, 4, 20]), torch.Size([2, 4, 20]))

In [38]:
torch.tril(torch.ones(4, 4), diagonal=0)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

In [40]:
a = torch.ones(4,4)
b = torch.tril(a, diagonal=0)

a.masked_fill(b==0, float('-inf'))

tensor([[1., -inf, -inf, -inf],
        [1., 1., -inf, -inf],
        [1., 1., 1., -inf],
        [1., 1., 1., 1.]])

In [43]:
torch.tensor([1,2,3,5]).numel(), torch.tensor([[1,2,3], [1,2,3]]).numel()

(4, 6)

In [49]:
nn.Parameter(torch.tensor([1,2,3], dtype=torch.float))

Parameter containing:
tensor([1., 2., 3.], requires_grad=True)

In [50]:
torch.randn(5,5)

tensor([[-0.2390,  0.2909,  0.6277,  0.2033,  1.3063],
        [ 0.6512, -1.0781,  0.7959,  1.1208,  0.0732],
        [ 1.0339, -0.7187,  1.0732, -1.8439, -1.2283],
        [-0.9979, -0.6068,  0.3396,  0.1857,  1.2969],
        [-0.5376,  0.9553, -1.7991, -0.2396,  1.1859]])

In [52]:
nn.Linear(5,5).weight.shape, nn.Parameter(torch.rand(5,5)).shape

(torch.Size([5, 5]), torch.Size([5, 5]))

In [53]:
nn.Linear(5,5).weight

Parameter containing:
tensor([[ 0.3686, -0.3447, -0.1195,  0.0856,  0.3903],
        [-0.2328, -0.3955,  0.0625,  0.4407, -0.0130],
        [-0.1590,  0.1403,  0.2101, -0.4384,  0.1777],
        [-0.0390,  0.2196, -0.0005, -0.3508,  0.3092],
        [ 0.1379,  0.1337, -0.0059, -0.2326, -0.0905]], requires_grad=True)

In [54]:
nn.Parameter(torch.rand(5,5))

Parameter containing:
tensor([[0.5082, 0.2248, 0.7780, 0.5903, 0.5948],
        [0.2635, 0.3252, 0.6510, 0.2531, 0.8348],
        [0.1452, 0.9151, 0.0598, 0.1477, 0.8348],
        [0.7600, 0.3244, 0.2850, 0.9895, 0.9097],
        [0.7428, 0.9225, 0.7265, 0.4033, 0.6398]], requires_grad=True)

In [62]:
class MultiTaskModel(nn.Module):
    def __init__(self,):
        super().__init__()
        self.register_buffer('age_mean', torch.tensor(0.0))
        self.register_buffer('age_std', torch.tensor(1.0))
        self.test_age_mean = torch.tensor(0.0)
        self.test_age_std = torch.tensor(1.0)

In [63]:
mtm = MultiTaskModel()

In [64]:
mtm.age_mean

tensor(0.)

In [65]:
mtm.state_dict()

OrderedDict([('age_mean', tensor(0.)), ('age_std', tensor(1.))])

In [59]:
torch.triu(torch.ones(5,5))

tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])

In [60]:
torch.tril(torch.ones(5,5))

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [81]:
torch.randn(2,3,5).shape

torch.Size([2, 3, 5])

In [82]:
torch.randn(2,3,5).unsqueeze(0).shape

torch.Size([1, 2, 3, 5])

In [83]:
torch.randn(2,3,5).unsqueeze(0).unsqueeze(2).shape

torch.Size([1, 2, 1, 3, 5])

In [90]:
torch.randn(2,3,5).transpose(-2, -1).shape

torch.Size([2, 5, 3])

In [91]:
test = torch.randn(4, 8, 10)
b, t, d = test.shape
b,t,d

(4, 8, 10)

In [95]:
d // 2

5

In [96]:
n_heads = 2
test.view(b, t, n_heads, d // n_heads).shape

torch.Size([4, 8, 2, 5])

In [97]:
test.view(b, t, n_heads, d // n_heads).view(b, t, d).shape

torch.Size([4, 8, 10])

In [ ]:
(test != test.view(b, t, n_heads, d // n_heads).view(b, t, d)).sum()

tensor(0)

In [102]:
torch.arange(20).view(4, 5)

tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])

In [103]:
torch.arange(20).view(5, 4)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])

In [107]:
test = torch.rand(2, 4, 3)
test.shape

torch.Size([2, 4, 3])

In [109]:
test.sum(dim=-1).shape

torch.Size([2, 4])

In [110]:
test.sum(dim=-1)

tensor([[1.3958, 0.8370, 2.0361, 1.5005],
        [0.9551, 1.3621, 2.0611, 1.5937]])

In [113]:
test.sum(dim=1, keepdim=True).shape

torch.Size([2, 1, 3])

In [115]:
test.sum(dim=1, keepdim=False)

tensor([[1.8382, 1.9967, 1.9344],
        [2.4281, 1.6449, 1.8990]])

In [114]:
test.sum(dim=1, keepdim=True)

tensor([[[1.8382, 1.9967, 1.9344]],

        [[2.4281, 1.6449, 1.8990]]])